# 04 · 把数字任务换成德英翻译

上一篇让模型倒着写数字，这次让它读一句德语，写出一句英语。模型仍然做熟悉的事：**Encoder 读输入，Decoder 接着已有前缀写下一词。**

真正新增的工作发生在两端：进模型前，把文字变成 token 编号；出模型后，把编号还原成文字。我们沿着一条句子的旅程，把分词、词表、补齐、训练和生成连起来。

## 准备语料和分词器

在 `transformer` 目录执行：

```bash
uv sync --python 3.11.15 --extra translation
uv run python -m spacy download de_core_news_sm
uv run python -m spacy download en_core_web_sm
uv run jupyter lab
```

选择 `transformer/.venv/bin/python` 内核。这篇使用 torchtext 0.18、torch 2.3 和 Multi30k 德英平行语料，首次准备数据需要联网。

`RUN_TRANSLATION=False` 是阅读模式；准备好依赖后改为 `True`，重启内核并顺序运行，就会构建词表并开始训练。默认五轮，可以先调小模型宽度和训练轮数体验流程。

<details>
<summary>下载或安装遇到问题时，从哪里查起？</summary>

先区分出错环节：可选 Python 包、spaCy 语言模型、Multi30k 语料分别下载。torchtext 是旧版本组合，其平台支持与语料链接的可用性需要以当前运行结果为准。

</details>

In [ ]:
import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_TRANSLATION = False
BATCH_SIZE = 16
NUM_EPOCHS = 5
SRC_LANGUAGE, TGT_LANGUAGE = "de", "en"
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ["<unk>", "<pad>", "<bos>", "<eos>"]
print("Device:", DEVICE, "Translation enabled:", RUN_TRANSLATION)

## 沿用前两篇的模型积木

先运行下面四组定义，把完整 Transformer 准备好。学过前几篇后，可以折叠这些代码，直接从“分词与词表”开始阅读。

本篇仍使用 Post-LN 和输出对数概率的 Generator；训练函数会与它配套使用 `nll_loss`。

### 注意力：译文带着问题查原文

Decoder 的 Q 来自当前译文前缀，交叉注意力的 K/V 来自德语 memory。两种语言的 token 数可以不同，所以 Q 的长度和 K/V 的长度分别处理。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        Q = self.w_q(q).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        K = self.w_k(k).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        V = self.w_v(v).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.dropout(F.softmax(scores, dim=-1))
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.w_o(context)

### FFN 与连接层：把收集来的信息加工好

FFN 逐位置处理特征，残差与 LayerNorm 把各子层接起来。无论句子长短，每个位置都使用同一套 FFN 参数。

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.w_2(self.dropout(F.relu(self.w_1(x))))


class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        variance = x.var(-1, keepdim=True, unbiased=False)
        return self.a_2 * (x - mean) / torch.sqrt(variance + self.eps) + self.b_2


class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, sublayer):
        return self.norm(x + self.dropout(sublayer(x)))

### 两端的接口：文字编号进，英语词表分数出

源侧和目标侧分别有自己的 Embedding，位置编码补充次序。Generator 的输出宽度等于英语词表大小：每一列对应一个候选英语 token。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        self.pe: torch.Tensor
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model
    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)


class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)
    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)

### Encoder 与 Decoder：让两种语言在 memory 处碰面

Encoder 先读完德语句子，Decoder 每一步读取英语前缀并查询德语 memory。`make_model` 把这条路径连起来，接下来给它准备真正的输入。

In [ ]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])


class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size
    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)


class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)
    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)


class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask)
        output = self.decode(memory, src_mask, tgt, tgt_mask)
        return output

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)


def make_model(src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1):
    c = copy.deepcopy
    attn = MultiHeadAttention(d_model, h)
    ff = PositionWiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = Transformer(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab)
    )
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model

## 1. 给文字编号：分词与词表

想象先把句子剪成词片，再给每种词片分配一个号码：

```text
一条德语句子 → 分词器 → token 列表 → 德语词表 → 整数列表
一条英语句子 → 分词器 → token 列表 → 英语词表 → 整数列表
```

词表从训练语料里收集，遇到词表外的 token 就使用 UNK 编号。下面分别建立两种语言的词表，最后打印各自有多少个条目。

Multi30k 通过迭代器逐条提供句子对。构建每个词表时，代码都创建一份新的迭代器，从头读取训练语料。

In [ ]:
def yield_tokens(data_iter, language):
    column = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1}[language]
    for sample in data_iter:
        yield tokenizers[language](sample[column])

if RUN_TRANSLATION:
    import torchtext
    torchtext.disable_torchtext_deprecation_warning()
    from torchtext.data.utils import get_tokenizer
    from torchtext.vocab import build_vocab_from_iterator
    from torchtext.datasets import Multi30k

    tokenizers = {
        SRC_LANGUAGE: get_tokenizer("spacy", language="de_core_news_sm"),
        TGT_LANGUAGE: get_tokenizer("spacy", language="en_core_web_sm"),
    }
    vocab_transform = {}
    for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
        iterator = Multi30k(split="train", language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
        vocab_transform[language] = build_vocab_from_iterator(
            yield_tokens(iterator, language), min_freq=1,
            specials=special_symbols, special_first=True,
        )
        vocab_transform[language].set_default_index(UNK_IDX)
    print("Vocabulary sizes:", {key: len(value) for key, value in vocab_transform.items()})

## 2. 句子有长有短，怎样装进同一个 batch？

给每句加上起始符 BOS 和结束符 EOS，再把短句补到当前 batch 的最长长度：

```text
BOS  I   run   EOS  PAD
BOS  I   love  AI   EOS
```

这样两句就能排进同一个矩形张量。BOS 告诉模型从哪里起步，EOS 表示一句写完了，PAD 负责填齐空位。

`text_to_tensor` 完成编号与首尾符号，`collate_fn` 收集一批句子，`pad_sequence` 把它们排整齐。

In [ ]:
def text_to_tensor(text, language):
    tokens = tokenizers[language](text.rstrip("\n"))
    ids = vocab_transform[language](tokens)
    return torch.tensor([BOS_IDX, *ids, EOS_IDX], dtype=torch.long)

def collate_fn(batch):
    sources = [text_to_tensor(source, SRC_LANGUAGE) for source, target in batch]
    targets = [text_to_tensor(target, TGT_LANGUAGE) for source, target in batch]
    return (
        pad_sequence(sources, batch_first=True, padding_value=PAD_IDX),
        pad_sequence(targets, batch_first=True, padding_value=PAD_IDX),
    )

## 3. 一张表挡住空位，一张表挡住未来

padding mask 标记有效 token，让注意力把份额分给句子内容；因果 mask 则让每个目标位置读取已有前缀。

源侧 mask 的形状是 `(B,1,1,Ls)`，其中两个 1 方便它共享给各个头和 Query。目标侧把 padding mask 与 `(1,1,Lt,Lt)` 的下三角表合起来。

训练输入依旧右移一位：`tgt[:, :-1]` 作为输入，`tgt[:, 1:]` 作为标签。`nll_loss(ignore_index=PAD_IDX)` 在计分时跳过补齐位置。

In [ ]:
def create_mask(src, tgt):
    src_mask = (src != PAD_IDX)[:, None, None, :]
    target_padding = (tgt != PAD_IDX)[:, None, None, :]
    causal = torch.ones(tgt.size(1), tgt.size(1), dtype=torch.bool, device=tgt.device).tril()
    return src_mask, target_padding & causal[None, None]

def train_epoch(model, dataloader, optimizer):
    model.train()
    loss_total, batch_count = 0.0, 0
    for src, tgt in dataloader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        decoder_input, labels = tgt[:, :-1], tgt[:, 1:]
        src_mask, tgt_mask = create_mask(src, decoder_input)
        hidden = model(src, decoder_input, src_mask, tgt_mask)
        log_probs = model.generator(hidden)
        loss = F.nll_loss(log_probs.reshape(-1, log_probs.size(-1)), labels.reshape(-1), ignore_index=PAD_IDX)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_total += loss.item()
        batch_count += 1
    return loss_total / max(batch_count, 1)

## 4. 开始读句子、写译文

开关开启后，下面创建三层、隐藏宽度为 512 的模型。每轮从训练语料取出一批批句子对，计算下一词损失，再更新参数。

观察打印的 NLL：它衡量模型在正确英语前缀下，给正确下一词分配了多少概率。CPU 上可先用更小的宽度和轮数，熟悉一次完整训练需要多长时间。

In [ ]:
if RUN_TRANSLATION:
    translator = make_model(
        len(vocab_transform[SRC_LANGUAGE]), len(vocab_transform[TGT_LANGUAGE]),
        N=3, d_model=512, d_ff=512, h=8, dropout=0.1,
    ).to(DEVICE)
    optimizer = torch.optim.Adam(translator.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
    for epoch in range(NUM_EPOCHS):
        train_data = Multi30k(split="train", language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
        dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, collate_fn=collate_fn)
        loss = train_epoch(translator, dataloader, optimizer)
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}: NLL={loss:.4f}")
else:
    print("Translation is disabled; no data downloaded and no model trained.")

## 5. 让模型写完一句英语

把德语句子送进 Encoder，得到 memory；然后从 BOS 起步，逐词追加预测。遇到 EOS，或达到 20 个新 token，就结束。

这和倒序任务是同一段生成逻辑。区别在于最后一步：通过英语词表把编号变回词，用空格拼起来显示。

读生成结果时，可以先看主语、动作和句子结尾，再回头看看德语原句中对应的信息。

In [ ]:
@torch.no_grad()
def translate(model, sentence, max_new_tokens=20):
    model.eval()
    src = text_to_tensor(sentence, SRC_LANGUAGE)[None].to(DEVICE)
    src_mask = (src != PAD_IDX)[:, None, None, :]
    memory = model.encode(src, src_mask)
    prefix = torch.full((1, 1), BOS_IDX, dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        _, tgt_mask = create_mask(src, prefix)
        hidden = model.decode(memory, src_mask, prefix, tgt_mask)
        next_token = model.generator(hidden[:, -1]).argmax(dim=-1, keepdim=True)
        prefix = torch.cat([prefix, next_token], dim=1)
        if next_token.item() == EOS_IDX:
            break
    ids = [idx for idx in prefix[0].tolist() if idx not in (BOS_IDX, EOS_IDX, PAD_IDX)]
    return " ".join(vocab_transform[TGT_LANGUAGE].lookup_tokens(ids))

if RUN_TRANSLATION:
    sentence = "Eine Gruppe von Menschen steht vor einem Iglu ."
    print("German:", sentence)
    print("Generated English:", translate(translator, sentence))

## 一条句子走完了全程

从文字到整数，从整数到向量，再从隐藏状态回到词表概率：前面的注意力、位置编码和训练机制，现在都接到了真实文本上。

还可以尝试给德语句子换一个名词或动作，观察译文的哪些位置随之改变。进一步学习解码时，贪心每步选最高分，beam search 保留多条累计得分候选，采样则按分布抽取。

返回[Transformer 学习入口](01_注意力与多头机制.ipynb)。